Original full-data pipeline recipe, retained for provenance. The deployed artifact uses this configuration. Prefer python -m training.train --fit-all for a clean executable entry point; do not replace the deployed artifact without validation.


In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "backend").is_dir() and (p / "data").is_dir())
sys.path.insert(0, str(ROOT))


In [ ]:
output_dir = ROOT / "training/runs/notebook"
output_dir.mkdir(parents=True, exist_ok=True)
import joblib
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, FunctionTransformer
from xgboost import XGBRegressor

# ✅ Import your custom components
from backend.app.preprocessing import binary_map, FrequencyEncoder

# 1. Load data
df = pd.read_csv(ROOT / "data/processed/cars_preprocessed.csv")
y = df["Price"]
X = df.drop(columns=["Price"])

# 2. Define column types
binary_cols = ['Première_main', 'Transmission']
ordinal_col = ['État']
ordinal_order = [['mauvais', 'correct', 'bon', 'très bon', 'excellent', 'neuf']]
onehot_cols = ['Carburant', 'Origine']
freq_cols = ['Marque', 'Model']
passthrough_cols = ['Nb_portes']
numerical_cols = ['Kilométrage', 'Puissance']

# Optional: create Car_Age if needed
X["Car_Age"] = 2025 - X["Model_Year"]
numerical_cols.append("Car_Age")
X.drop(columns=["Model_Year"], inplace=True)

# 3. Column Transformer
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numerical_cols),
    ('binary', FunctionTransformer(binary_map), binary_cols),
    ('ordinal', OrdinalEncoder(categories=ordinal_order), ordinal_col),
    ('onehot', OneHotEncoder(handle_unknown='ignore'), onehot_cols),
    ('freq', FrequencyEncoder(), freq_cols),
    ('pass', 'passthrough', passthrough_cols)
])

# 4. Final pipeline
pipeline = Pipeline(steps=[
    ('prep', preprocessor),
    ('model', XGBRegressor(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.5,
        reg_lambda=2,
        random_state=42
    ))
])

# 5. Fit the model
pipeline.fit(X, y)

# 6. Save pipeline
joblib.dump(pipeline, output_dir / "pipeline.joblib", compress=3)
print("✅ Pipeline saved successfully!")
